In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

data = pd.read_csv("BreastCancer.csv")

# Convert diagnosis to numerical values
data["diagnosis_numeric"] = data["diagnosis"].map({"B": 0, "M": 1})

print(data.head())

         id diagnosis  radius_mean  texture_mean  perimeter_mean  area_mean  \
0    842302         M        17.99         10.38          122.80     1001.0   
1    842517         M        20.57         17.77          132.90     1326.0   
2  84300903         M        19.69         21.25          130.00     1203.0   
3  84348301         M        11.42         20.38           77.58      386.1   
4  84358402         M        20.29         14.34          135.10     1297.0   

   smoothness_mean  compactness_mean  concavity_mean  concave points_mean  \
0          0.11840           0.27760          0.3001              0.14710   
1          0.08474           0.07864          0.0869              0.07017   
2          0.10960           0.15990          0.1974              0.12790   
3          0.14250           0.28390          0.2414              0.10520   
4          0.10030           0.13280          0.1980              0.10430   

   ...  texture_worst  perimeter_worst  area_worst  smoothness

In [ ]:
def Desc_stats(ds, var):
    # Returns detailed descriptive statistics for a given variable
    mean = ds[var].mean()
    median = ds[var].median()
    mode = ds[var].mode()[0]
    minimum = ds[var].min()
    maximum = ds[var].max()
    range_var = maximum - minimum
    std_dev = ds[var].std()
    variance = ds[var].var()
    skew = ds[var].skew()
    kurtosis = ds[var].kurt()

    count = ds[var].count()

    q1 = ds[var].quantile(0.25)
    q2 = ds[var].quantile(0.50)
    q3 = ds[var].quantile(0.75)
    iqr = q3 - q1

    summary = {
        "Count": count,
        "Mean": mean,
        "Median": median,
        "Mode": mode,
        "Minimum": minimum,
        "Maximum": maximum,
        "Range": range_var,
        "Standard Deviation": std_dev,
        "Variance": variance,
        "Skewness": skew,
        "Kurtosis": kurtosis,
        "25th Percentile (Q1)": q1,
        "50th Percentile (Median/Q2)": q2,
        "75th Percentile (Q3)": q3,
        "Interquartile Range (IQR)": iqr
    }

    return summary

In [ ]:
radius_stats = Desc_stats(data, "radius_mean")

for measure, value in radius_stats.items():
    print(measure, ":", value)

In [ ]:
sample_size = 150

sample_data = data.sample(
    n=sample_size,
    random_state=42
)

sample_stats = Desc_stats(
    sample_data,
    "diagnosis_numeric"
)

for measure, value in sample_stats.items():
    print(measure, ":", value)

In [ ]:
sample_size = 150
step = 3

np.random.seed(42)
start = np.random.randint(0, step)

positions = [
    (start + i * step) % len(data)
    for i in range(sample_size)
]

systematic_sample = data.iloc[positions]

systematic_stats = Desc_stats(
    systematic_sample,
    "diagnosis_numeric"
)

for measure, value in systematic_stats.items():
    print(measure, ":", value)

In [ ]:
plt.figure(figsize=(8, 5))

for diagnosis, color in [("B", "steelblue"), ("M", "coral")]:
    group = data[data["diagnosis"] == diagnosis]

    plt.scatter(
        group["radius_mean"],
        group["texture_mean"],
        color=color,
        alpha=0.6,
        label="Benign" if diagnosis == "B" else "Malignant"
    )

plt.xlabel("Mean radius")
plt.ylabel("Mean texture")
plt.title("Scatter: mean radius vs mean texture, grouped by diagnosis")
plt.legend(title="Diagnosis")
plt.tight_layout()
plt.show()

In [ ]:
benign = data[data["diagnosis"] == "B"]["radius_mean"]
malignant = data[data["diagnosis"] == "M"]["radius_mean"]

plt.figure(figsize=(8, 5))

plt.boxplot(
    [benign, malignant],
    labels=["Benign (B)", "Malignant (M)"]
)

plt.ylabel("Mean radius")
plt.title("Box plot: mean radius by diagnosis")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    data["diagnosis_numeric"],
    bins=[-0.5, 0.5, 1.5],
    edgecolor="white"
)

plt.xticks(
    [0, 1],
    ["Benign (B=0)", "Malignant (M=1)"]
)

plt.xlabel("Diagnosis")
plt.ylabel("Number of cases")
plt.title("Histogram of diagnosis (binary encoded outcomes)")
plt.tight_layout()
plt.show()

In [ ]:
columns = [
    "diagnosis_numeric",
    "radius_mean",
    "texture_mean",
    "perimeter_mean",
    "area_mean"
]

correlation_matrix = data[columns].corr()

plt.figure(figsize=(8, 6))

plt.imshow(
    correlation_matrix,
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)

plt.xticks(
    range(len(columns)),
    columns,
    rotation=45,
    ha="right"
)

plt.yticks(range(len(columns)), columns)

plt.colorbar(label="Pearson correlation")

plt.title("Heat map including encoded diagnosis")
plt.tight_layout()
plt.show()

In [ ]:
# Create three radius groups
data["radius_group"] = pd.qcut(
    data["radius_mean"],
    q=3,
    labels=["Low", "Medium", "High"]
)

observed = pd.crosstab(
    data["radius_group"],
    data["diagnosis"]
)

print("Observed Frequency Table:")
print(observed)

# Perform Chi-Square Test
chi2, p_value, dof, expected = stats.chi2_contingency(
    observed
)

print("Chi-Square Statistic:", round(chi2, 4))
print("Degrees of Freedom:", dof)
print("P-value:", p_value)

# Decision
if p_value < 0.05:
    print("Reject H0")
else:
    print("Fail to reject H0")

In [ ]:
from scipy import stats

sample_values = sample_data["diagnosis_numeric"]

# Perform one-sample t-test
t_stat, p_value = stats.ttest_1samp(
    sample_values,
    popmean=0.50
)

print("T-statistic:", t_stat)
print("P-value:", p_value)

# Hypothesis testing decision
if p_value < 0.05:
    print("Reject H0")
else:
    print("Fail to reject H0")